# Group 10 — M1 Colab CPU-only pilot

Notebook vận hành **3 ID đã có checkpoint Discovery resolved trong pilot local**;
đây không phải kết quả Colab đã chạy. Chọn **Runtime → Change runtime type → None**.
Đọc [hướng dẫn bàn giao](../docs/colab.md) trước khi chạy. Không dùng Run all để
bắt đầu crawl: hai ô chạy mạng mặc định tắt và phải được bật riêng sau khi review.

Code ở `/content`, corpus/logs/reports trên Drive riêng. Chỉ **một writer** trên
cùng corpus, kể cả notebook/máy khác. Dừng runtime cũ trước khi Resume.
S2 tắt; mọi version được Discovery xác nhận đều được xử lý, có revision history.
Không có Full Crawl, S2 batching hoặc exporter dataset trong notebook này.

In [ ]:
# Điền commit đầy đủ SAU KHI người dùng review và tự commit/push bộ bàn giao.
CODE_COMMIT = ""
REPOSITORY_URL = "https://github.com/Toan-ten/Arxiv-Project-Group10.git"
PAIRS = [("24120010", "2508.07872"), ("24120010", "2508.12871"),
         ("24120035", "2507.16659")]
COHORT_ID = "colab-arxiv-pilot-01"  # đổi tên cho cohort MỚI; giữ tên khi Resume
BATCH_ID = "arxiv-full-01"
CPU_ONLY_CONFIRMED = False         # xác nhận lựa chọn None trong giao diện Colab
SINGLE_WRITER_CONFIRMED = False    # bảo đảm mọi writer khác đã dừng
RUN_PILOT = False
RUN_RESUME = False

## 1. Mount Drive và tải đúng code

Colab sẽ yêu cầu quyền Drive. Không dùng dữ liệu Pilot cũ. Nếu mount/I/O thất bại,
dừng và xử lý Drive trước; không chuyển âm thầm sang `/content` cho checkpoint.
Mỗi runtime mới cần thực hiện lại setup. Cell clone chỉ đổi commit trong clone mới;
clone đã tồn tại phải sạch và đúng commit, không tự reset hoặc cập nhật nó.

In [ ]:
from pathlib import Path
import os
from google.colab import drive
drive.mount("/content/drive")
if not os.path.ismount("/content/drive"):
    raise RuntimeError("Drive mount chưa được xác nhận")
DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.is_dir():
    raise RuntimeError("Không tìm thấy MyDrive")

In [ ]:
import importlib.metadata
import json
import platform
import re
import subprocess
import sys

if not re.fullmatch(r"[0-9a-f]{40}", CODE_COMMIT):
    raise ValueError("Điền CODE_COMMIT đầy đủ đã review; không chạy theo latest")
if sys.version_info < (3, 10):
    raise RuntimeError("Cần Python >= 3.10; ghi nhận phiên bản Colab thực tế")
REPO = Path("/content/Arxiv-Project-Group10")
if not REPO.exists():
    subprocess.run(["git", "clone", "--single-branch", "--branch", "feat/m1-integration",
                    REPOSITORY_URL, str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_COMMIT], check=True)
def git_read(*args):
    return subprocess.check_output(["git", "-C", str(REPO), *args], text=True).strip()
if git_read("rev-parse", "HEAD") != CODE_COMMIT or git_read("status", "--porcelain"):
    raise RuntimeError("Clone không đúng commit hoặc đang có thay đổi; không tự ghi đè")
if git_read("remote", "get-url", "origin") != REPOSITORY_URL:
    raise RuntimeError("Sai repository origin")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements-colab.txt")], check=True)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print(json.dumps(dict(commit=CODE_COMMIT, python=platform.python_version(),
                     os=platform.platform(), psutil=importlib.metadata.version("psutil"))))
# Không đọc .env hay Colab Secrets ở phase arXiv-only này.

## 2. Kiểm tra filesystem trước khi gọi mạng

Probe sau chỉ dùng fixture được gắn nhãn, nằm ngoài corpus thật. Kiểm tra write,
fsync, replace, đọc JSON, regular-file/link count và SHA-256 tại Drive mount này.
Probe không chứng minh crash consistency hay độ bền khi Drive chưa đồng bộ.
Giữ probe để audit; không sửa/xóa checkpoint để ép Resume.

In [ ]:
import hashlib
import tempfile
from datetime import datetime, timezone
from uuid import uuid4
from src import arxiv_client, arxiv_history, benchmark_runner, pipeline
from src.storage import CorpusStorage, StorageError, _atomic_json

def validate_layout(repo, corpus, logs, reports, scratch):
    repo = Path(repo).resolve()
    paths = [Path(p).resolve() for p in (corpus, logs, reports, scratch)]
    for index, path in enumerate(paths):
        if path.is_relative_to(repo) or repo.is_relative_to(path):
            raise ValueError("Output phải hoàn toàn ngoài Git repository")
        if any(path.is_relative_to(p) or p.is_relative_to(path) for p in paths[:index]):
            raise ValueError("Corpus/logs/reports/scratch phải tách biệt")
    return paths

def validate_selection(pairs, corpus):
    if not isinstance(pairs, (list, tuple)) or not 1 <= len(pairs) <= 5:
        raise ValueError("Notebook pilot chỉ nhận 1–5 ID, không tự chạy Full Crawl")
    return pipeline._inputs(pairs, CorpusStorage(corpus), 30.0, 1, arxiv_client.USER_AGENT)

def drive_probe(directory):
    probe = Path(directory) / ("fixture-" + uuid4().hex)
    probe.mkdir(parents=True, exist_ok=False)
    store = CorpusStorage(probe)
    checkpoint = probe / "checkpoint.json"
    _atomic_json(checkpoint, dict(fixture=True, revision=1))
    _atomic_json(checkpoint, dict(fixture=True, revision=2))
    if store._read(checkpoint) != dict(fixture=True, revision=2):
        raise RuntimeError("Drive atomic JSON roundtrip thất bại")
    source = probe / "source" / "nested" / "main.tex"
    source.parent.mkdir(parents=True)
    payload = b"% OFFLINE FILESYSTEM FIXTURE ONLY\r\n\\input{part}\n"
    with source.open("xb") as stream:
        stream.write(payload)
        stream.flush()
        os.fsync(stream.fileno())
    files, checksums, total = store._snapshot(probe / "source")
    if (files != ["nested/main.tex"] or total != len(payload)
            or checksums[files[0]] != hashlib.sha256(payload).hexdigest()):
        raise RuntimeError("Drive source checksum roundtrip thất bại")
    result = dict(fixture=True, probe_path=str(probe), verified=True,
                  measured_at=datetime.now(timezone.utc).isoformat(),
                  checks=["fsync", "replace", "JSON read", "regular file", "SHA-256"],
                  limitation="not a crash-durability or remote-sync guarantee")
    _atomic_json(probe / "result.json", result)
    return result

def audit_cohort():
    audit = benchmark_runner.aggregate_batch(CORPUS, batch_id=BATCH_ID, log_root=LOGS)
    histories = []
    store = CorpusStorage(CORPUS)
    for pair in PAIRS:
        try:
            metadata = store.read_metadata(*pair)
            if metadata["version_status"] == "resolved":
                verified = arxiv_history._cached(metadata) or arxiv_history._single_version_cached(metadata)
            else:
                verified = False
            histories.append(dict(mssv=pair[0], arxiv_id=pair[1], verified=verified,
                dates=[dict(version=v["version"], date=v["date"], date_status=v["date_status"])
                       for v in metadata["versions"]]))
        except (StorageError, OSError, ValueError, KeyError, TypeError) as exc:
            histories.append(dict(mssv=pair[0], arxiv_id=pair[1], verified=False,
                                  error_type=type(exc).__name__))
    coverage = audit.get("data_coverage") or {}
    return dict(aggregation=audit, revision_history=histories,
        ready_for_zero_http_resume=(audit["status"] == "verified"
            and coverage.get("all_version_sources_terminal") is True
            and all(h["verified"] for h in histories)))

def protected_hashes():
    store = CorpusStorage(CORPUS)
    hashes = {}
    for pair in PAIRS:
        directory = store.paper_path(*pair)
        for path in directory.rglob("*"):
            if path.is_file() and (path.name in {"metadata.json", "references.json"}
                                   or path.suffix.lower() in {".tex", ".bib"}):
                hashes[str(path.relative_to(CORPUS))] = hashlib.sha256(path.read_bytes()).hexdigest()
    return hashes

def run_attempt(*, resume):
    if CPU_ONLY_CONFIRMED is not True or SINGLE_WRITER_CONFIRMED is not True:
        raise RuntimeError("Xác nhận CPU-only và writer cũ đã dừng trước khi chạy")
    if not PREFLIGHT.get("verified"):
        raise RuntimeError("Filesystem preflight chưa đạt")
    validate_layout(REPO, CORPUS, LOGS, REPORTS, SCRATCH)
    selected = validate_selection(PAIRS, CORPUS)
    if not resume and any(p.exists() and any(p.iterdir()) for p in (CORPUS, LOGS, REPORTS)):
        raise RuntimeError("Pilot đầu cần cohort mới/rỗng; dữ liệu đã có phải review và Resume")
    if git_read("rev-parse", "HEAD") != CODE_COMMIT or git_read("status", "--porcelain"):
        raise RuntimeError("Code đã đổi sau setup; dừng để review")
    experiment = "colab-" + uuid4().hex
    notes = COHORT / "sessions" / (experiment + ".json")
    notes.parent.mkdir(parents=True, exist_ok=True)
    _atomic_json(notes, dict(experiment_id=experiment, batch_id=BATCH_ID,
        recorded_at=datetime.now(timezone.utc).isoformat(), code_commit=CODE_COMMIT,
        runtime="Google Colab", accelerator="None (operator confirmed)",
        python=platform.python_version(), operating_system=platform.platform(),
        cpu_count=os.cpu_count(), psutil=importlib.metadata.version("psutil"),
        selection=selected, resume=resume, phase="arxiv_only",
        filesystem_probe=PREFLIGHT["probe_path"], corpus_root=str(CORPUS),
        scratch_root=str(SCRATCH), disk_scope="corpus plus dedicated download scratch",
        limitation="not the final full-M1 arxiv+S2 benchmark"))
    report = benchmark_runner.run_benchmark(selected, CORPUS, batch_id=BATCH_ID,
        max_ids=len(selected), log_root=LOGS, report_root=REPORTS,
        experiment_id=experiment, resume=resume, versions=None, revision_history=True,
        s2_config=None, timeout=30.0, max_attempts=1, sample_interval=0.1,
        disk_interval=30.0, disk_roots=[CORPUS, SCRATCH])
    print(json.dumps(dict(execution_status=report["execution_status"],
        batch_status=report["batch_status"], stop_reason=report["stop_reason"],
        report_path=report["report_path"], benchmark_metrics=report["benchmark_metrics"],
        timing_metrics=report["timing_metrics"], data_coverage=report["data_coverage"],
        http_metrics=report["aggregation"]["history"]["http_metrics"]), ensure_ascii=False, indent=2))
    if report["execution_status"] != "returned" or report["batch_status"] != "finished":
        raise RuntimeError("Batch dừng/lỗi: review report, không tự chạy lại")
    return report

In [ ]:
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}", COHORT_ID):
    raise ValueError("COHORT_ID không hợp lệ")
COHORT = (DRIVE_ROOT / "Group10-M1" / COHORT_ID).resolve()
CORPUS, LOGS, REPORTS = (COHORT / name for name in ("corpus", "logs", "reports"))
SCRATCH = Path("/content/group10-scratch") / COHORT_ID
validate_layout(REPO, CORPUS, LOGS, REPORTS, SCRATCH)
validate_selection(PAIRS, CORPUS)  # ownership + format + duplicate, trước HTTP
COHORT.mkdir(parents=True, exist_ok=True)
SCRATCH.mkdir(parents=True, exist_ok=True)
# Process-local override vì Colab có thể đã cache tempfile.gettempdir().
tempfile.tempdir = str(SCRATCH)
os.environ["TMPDIR"] = str(SCRATCH)
PREFLIGHT = drive_probe(COHORT / "preflight")
print(json.dumps(PREFLIGHT, ensure_ascii=False, indent=2))
print("Output roots:", CORPUS, LOGS, REPORTS, SCRATCH, sep="\n")

## 3. Chạy pilot arXiv-first — thao tác mạng thủ công

Chỉ bật `CPU_ONLY_CONFIRMED`, `SINGLE_WRITER_CONFIRMED`, rồi đổi `RUN_PILOT=True`
trong **ô dưới** sau khi đã kiểm tra selection/paths. Giữ False ở ô Resume.
Pacing dùng chung ≥3 giây; `max_attempts=1`, dừng 403/429 tại Discovery, Source
hoặc History. Số requests phụ thuộc số versions thực tế, không phải 2 requests/ID.
Source giới hạn 100 MiB/download, 200 MiB giải nén/version theo downloader hiện có;
chưa có quota tổng cohort. Nếu disk/Drive gần đầy hoặc có lỗi, dừng và kiểm tra.

In [ ]:
RUN_PILOT = False  # chỉ đổi ô này khi sẵn sàng gọi arXiv thật
if RUN_PILOT:
    RUN_PILOT = False  # không tự lặp attempt khi chạy lại cell
    first_report = run_attempt(resume=False)
else:
    print("Dry run: chưa gọi arXiv/S2. Bật RUN_PILOT trong ô này để chạy pilot.")

## 4. Integrity và báo cáo chỉ đọc corpus

`verified` nghĩa bằng chứng nhất quán, không phải mọi publication complete.
Xem missing/invalid checkpoint, trạng thái từng version, ngày/provenance, pending,
blocked, errors và sample status. S2 tắt nên references `not_attempted`; paper
vẫn có thể partial dù mọi source/history đã xác minh. Không đổi unknown thành 0.
Ô sau lưu bản audit JSON mới, không xuất papers/references dataset hay raw corpus.

In [ ]:
if (LOGS / BATCH_ID / "batch.json").is_file():
    audit = audit_cohort()
    destination = COHORT / "handoff" / ("audit-" + uuid4().hex + ".json")
    destination.parent.mkdir(parents=True, exist_ok=True)
    _atomic_json(destination, audit)
    print(json.dumps(audit, ensure_ascii=False, indent=2))
    print("Audit report:", destination)
else:
    print("Chưa có batch; không tạo kết quả crawl giả.")

## 5. Resume — cùng cohort, selection, batch ID và cấu hình

Đọc report trước, bảo đảm writer cũ dừng. Sau 403/429 phải review cooldown/chính
sách trước, không bật tự động Resume. Khi runtime mới, chạy lại setup/preflight;
bỏ qua ô Pilot và chỉ bật ô Resume. Experiment ID mới được sinh, log cũ giữ nguyên.
Nếu trước Resume mọi source/history đã verified, ô này yêu cầu **0 HTTP** và giữ
nguyên metadata/references/source. Nếu còn pending/lỗi retryable, Resume có thể gọi
mạng cho phần còn thiếu. Không có retry loop ngoài client.

In [ ]:
RUN_RESUME = False  # chỉ bật sau review, không dùng cùng RUN_PILOT
if RUN_RESUME:
    RUN_RESUME = False
    before = audit_cohort()
    hashes_before = protected_hashes()
    resume_report = run_attempt(resume=True)
    after = audit_cohort()
    if before["ready_for_zero_http_resume"]:
        if not after["ready_for_zero_http_resume"]:
            raise RuntimeError("Integrity sau Resume không còn hợp lệ; cần review")
        attempt_path = LOGS / BATCH_ID / "attempts" / (resume_report["attempt_ids"][0] + ".json")
        current_attempt = json.loads(attempt_path.read_text(encoding="utf-8"))
        metrics = current_attempt["http_metrics"]
        if metrics["status"] != "collected" or metrics["total"]["attempts"] != 0:
            raise RuntimeError("Resume dữ liệu đã verified có HTTP mới; cần review")
        if protected_hashes() != hashes_before:
            raise RuntimeError("Metadata/references/source đã thay đổi; cần review")
        print("Resume verified: 0 HTTP, metadata/references/source giữ nguyên bytes.")
    print("Chạy lại ô audit để lưu kết quả mới; manifest run_id/updated_at có thể thay đổi.")
else:
    print("Resume chưa được bật; không gọi API.")

## 6. Bàn giao A → C

Giữ toàn bộ cohort gồm corpus, logs/attempts/measurements, reports, sessions và
handoff audits trên Drive. C dùng JSON report/measurement thật, không screenshot
hoặc notebook outputs làm dữ liệu chuẩn. Trước copy/sao lưu phải dừng writer;
giữ nguyên bản gốc, không move dữ liệu đang ghi. Xem `docs/colab.md` về cohort mới,
S2 enrichment bằng batch ID riêng và công thức tổng hợp nhiều phase/cohort.

Notebook này chỉ benchmark phase arXiv-only. Benchmark M1 cuối phải bao gồm cả
arXiv và S2 trên Colab CPU-only, có provenance và công bố thiếu phép đo do hard kill.